# Reinforcement Learning: Sim2Sim Transfer Data Visualization
Using a policy trained in IsaacLab, deploy to Mujoco on the same robot platform (AnyMAL C).

In [5]:
# Check the model parameters
import torch

DEFAULT_POLICY_PATH = "/home/lorin-cairo/Documents/AI-ML-Practice/IsaacLab/logs/rsl_rl/anymal_c_flat/2025-10-22_13-59-16/exported/policy.pt"

def count_policy_params(policy_path):
    """Count parameters in a PyTorch policy (handles both TorchScript and state_dict)."""
    try:
        # First try loading as TorchScript (exported policies from IsaacLab are TorchScript)
        policy = torch.jit.load(policy_path, map_location='cpu')
        total_params = sum(p.numel() for p in policy.parameters())
        print("✓ Loaded as TorchScript module")
        return total_params
    except Exception as e1:
        # Fall back to torch.load for state_dict format
        try:
            # PyTorch 2.6+ requires weights_only=False for TorchScript archives
            state = torch.load(policy_path, map_location='cpu', weights_only=False)
            if isinstance(state, dict):
                # If state_dict has 'model', it's from RLlib style save
                if 'model' in state:
                    parameters = state['model'].values()
                # If dict keys seem like weights, treat as state_dict
                else:
                    parameters = state.values()
                total_params = sum(p.numel() if hasattr(p, 'numel') else 0 for p in parameters)
                print("✓ Loaded as state_dict")
            else:
                # Loaded as a module
                total_params = sum(p.numel() for p in state.parameters())
                print("✓ Loaded as PyTorch module")
            return total_params
        except Exception as e2:
            print(f"Failed to load policy:")
            print(f"  TorchScript error: {e1}")
            print(f"  torch.load error: {e2}")
            return None

n_params = count_policy_params(DEFAULT_POLICY_PATH)
print(f"Number of parameters in the loaded policy: {n_params:,}" if n_params else "Failed to load policy")

✓ Loaded as TorchScript module
Number of parameters in the loaded policy: 40,844


In [7]:
# Print network structure
import torch

def print_network_structure(policy_path):
    """Display the neural network architecture."""
    print("="*70)
    print("NEURAL NETWORK ARCHITECTURE")
    print("="*70)
    
    # Load policy
    try:
        policy = torch.jit.load(policy_path, map_location='cpu')
    except:
        policy = torch.load(policy_path, map_location='cpu', weights_only=False)
    
    # Method 1: Print the model structure
    print("\n1. MODEL STRUCTURE:")
    print("-" * 70)
    print(policy)
    
    # Method 2: Show parameter details (layer names and shapes)
    print("\n2. LAYER DETAILS:")
    print("-" * 70)
    print(f"{'Layer Name':<40} {'Shape':<20} {'Parameters':>10}")
    print("-" * 70)
    
    total_params = 0
    for name, param in policy.named_parameters():
        num_params = param.numel()
        total_params += num_params
        print(f"{name:<40} {str(list(param.shape)):<20} {num_params:>10,}")
    
    print("-" * 70)
    print(f"{'TOTAL PARAMETERS:':<60} {total_params:>10,}")
    print("=" * 70)
# Print the network structure
print_network_structure(DEFAULT_POLICY_PATH)


NEURAL NETWORK ARCHITECTURE

1. MODEL STRUCTURE:
----------------------------------------------------------------------
RecursiveScriptModule(
  original_name=_TorchPolicyExporter
  (actor): RecursiveScriptModule(
    original_name=MLP
    (0): RecursiveScriptModule(original_name=Linear)
    (1): RecursiveScriptModule(original_name=ELU)
    (2): RecursiveScriptModule(original_name=Linear)
    (3): RecursiveScriptModule(original_name=ELU)
    (4): RecursiveScriptModule(original_name=Linear)
    (5): RecursiveScriptModule(original_name=ELU)
    (6): RecursiveScriptModule(original_name=Linear)
  )
  (normalizer): RecursiveScriptModule(original_name=Identity)
)

2. LAYER DETAILS:
----------------------------------------------------------------------
Layer Name                               Shape                Parameters
----------------------------------------------------------------------
actor.0.weight                           [128, 48]                 6,144
actor.0.bias               

In [ ]:
# IsaacLab Replay

import pandas as pd
import matplotlib.pyplot as plt

# ---- CONFIG ----
CSV_FILE = "isaaclab_log_20251105_112725.csv"  # Change to your actual file path
SHOW_FIG = True              # Set False if you just want to save the figures
SAVE_FIG = False              # Saves all plots as PNGs
# ----------------

# Read CSV
df = pd.read_csv(CSV_FILE)

# --- BASE POSITION ---
plt.figure(figsize=(10, 5))
plt.plot(df['time'], df['base_pos_x'], label='base_pos_x')
plt.plot(df['time'], df['base_pos_y'], label='base_pos_y')
plt.plot(df['time'], df['base_pos_z'], label='base_pos_z')
plt.title("Base Position Over Time")
plt.xlabel("Time [s]")
plt.ylabel("Position [m]")
plt.legend()
plt.grid(True)
if SAVE_FIG: plt.savefig("base_position.png", dpi=200, bbox_inches='tight')

# --- BASE ORIENTATION (QUATERNION) ---
plt.figure(figsize=(10, 5))
plt.plot(df['time'], df['base_quat_w'], label='quat_w')
plt.plot(df['time'], df['base_quat_x'], label='quat_x')
plt.plot(df['time'], df['base_quat_y'], label='quat_y')
plt.plot(df['time'], df['base_quat_z'], label='quat_z')
plt.title("Base Orientation (Quaternion) Over Time")
plt.xlabel("Time [s]")
plt.ylabel("Quaternion Value")
plt.legend()
plt.grid(True)
if SAVE_FIG: plt.savefig("base_orientation.png", dpi=200, bbox_inches='tight')

# --- JOINT POSITIONS ---
joint_pos_cols = [col for col in df.columns if col.startswith('joint_pos_')]
plt.figure(figsize=(12, 6))
for col in joint_pos_cols:
    plt.plot(df['time'], df[col], label=col)
plt.title("Joint Positions Over Time")
plt.xlabel("Time [s]")
plt.ylabel("Joint Position [rad or m]")
plt.legend(ncol=3, fontsize='small')
plt.grid(True)
if SAVE_FIG: plt.savefig("joint_positions.png", dpi=200, bbox_inches='tight')

# --- JOINT VELOCITIES ---
joint_vel_cols = [col for col in df.columns if col.startswith('joint_vel_')]
plt.figure(figsize=(12, 6))
for col in joint_vel_cols:
    plt.plot(df['time'], df[col], label=col)
plt.title("Joint Velocities Over Time")
plt.xlabel("Time [s]")
plt.ylabel("Joint Velocity [rad/s]")
plt.legend(ncol=3, fontsize='small')
plt.grid(True)
if SAVE_FIG: plt.savefig("joint_velocities.png", dpi=200, bbox_inches='tight')

# --- CONTACT FORCES ---
contact_force_cols = [col for col in df.columns if col.startswith('contact_force_')]

# Group by limb (LF, RF, LH, RH)
limbs = sorted(set(col.split('_')[2] for col in contact_force_cols))  # ['LF', 'LH', 'RF', 'RH']

for limb in limbs:
    plt.figure(figsize=(10, 5))
    cols = [col for col in contact_force_cols if f'_{limb}_' in col]
    for col in cols:
        plt.plot(df['time'], df[col], label=col)
    plt.title(f"Contact Forces ({limb}) Over Time")
    plt.xlabel("Time [s]")
    plt.ylabel("Force [N]")
    plt.legend()
    plt.grid(True)
    if SAVE_FIG:
        plt.savefig(f"contact_forces_{limb}.png", dpi=200, bbox_inches='tight')


if SHOW_FIG:
    plt.show()


In [ ]:
# Mujoco Replay

import pandas as pd
import matplotlib.pyplot as plt

# ---- CONFIG ----
CSV_FILE = "mujoco_single_robot_log.csv"  # Change to your actual file path
SHOW_FIG = True              # Set False if you just want to save the figures
SAVE_FIG = False              # Saves all plots as PNGs
# ----------------

# Read CSV
df = pd.read_csv(CSV_FILE)
df = df[df['time'].between(0, 40)]  # Filter to match IsaacLab replay data since we have more than 40 seconds of data

# --- BASE POSITION ---
plt.figure(figsize=(10, 5))
plt.plot(df['time'], df['base_pos_x'], label='base_pos_x')
plt.plot(df['time'], df['base_pos_y'], label='base_pos_y')
plt.plot(df['time'], df['base_pos_z'], label='base_pos_z')
plt.title("Base Position Over Time")
plt.xlabel("Time [s]")
plt.ylabel("Position [m]")
plt.legend()
plt.grid(True)
if SAVE_FIG: plt.savefig("base_position.png", dpi=200, bbox_inches='tight')

# --- BASE ORIENTATION (QUATERNION) ---
plt.figure(figsize=(10, 5))
plt.plot(df['time'], df['base_quat_w'], label='quat_w')
plt.plot(df['time'], df['base_quat_x'], label='quat_x')
plt.plot(df['time'], df['base_quat_y'], label='quat_y')
plt.plot(df['time'], df['base_quat_z'], label='quat_z')
plt.title("Base Orientation (Quaternion) Over Time")
plt.xlabel("Time [s]")
plt.ylabel("Quaternion Value")
plt.legend()
plt.grid(True)
if SAVE_FIG: plt.savefig("base_orientation.png", dpi=200, bbox_inches='tight')

# --- JOINT POSITIONS ---
joint_pos_cols = [col for col in df.columns if col.startswith('joint_pos_')]
plt.figure(figsize=(12, 6))
for col in joint_pos_cols:
    plt.plot(df['time'], df[col], label=col)
plt.title("Joint Positions Over Time")
plt.xlabel("Time [s]")
plt.ylabel("Joint Position [rad or m]")
plt.legend(ncol=3, fontsize='small')
plt.grid(True)
if SAVE_FIG: plt.savefig("joint_positions.png", dpi=200, bbox_inches='tight')

# --- JOINT VELOCITIES ---
joint_vel_cols = [col for col in df.columns if col.startswith('joint_vel_')]
plt.figure(figsize=(12, 6))
for col in joint_vel_cols:
    plt.plot(df['time'], df[col], label=col)
plt.title("Joint Velocities Over Time")
plt.xlabel("Time [s]")
plt.ylabel("Joint Velocity [rad/s]")
plt.legend(ncol=3, fontsize='small')
plt.grid(True)
if SAVE_FIG: plt.savefig("joint_velocities.png", dpi=200, bbox_inches='tight')

# --- CONTACT FORCES ---
contact_force_cols = [col for col in df.columns if col.startswith('contact_force_')]

# Group by limb (LF, RF, LH, RH)
limbs = sorted(set(col.split('_')[2] for col in contact_force_cols))  # ['LF', 'LH', 'RF', 'RH']

for limb in limbs:
    plt.figure(figsize=(10, 5))
    cols = [col for col in contact_force_cols if f'_{limb}_' in col]
    for col in cols:
        plt.plot(df['time'], df[col], label=col)
    plt.title(f"Contact Forces ({limb}) Over Time")
    plt.xlabel("Time [s]")
    plt.ylabel("Force [N]")
    plt.legend()
    plt.grid(True)
    if SAVE_FIG:
        plt.savefig(f"contact_forces_{limb}.png", dpi=200, bbox_inches='tight')


if SHOW_FIG:
    plt.show()


In [ ]:
# Mujoco Replay

import pandas as pd
import matplotlib.pyplot as plt

# ---- CONFIG ----
CSV_FILE = "force_mujoco_single_robot_log.csv"  # Change to your actual file path
SHOW_FIG = True              # Set False if you just want to save the figures
SAVE_FIG = False              # Saves all plots as PNGs
# ----------------

# Read CSV
df = pd.read_csv(CSV_FILE)
df = df[df['time'].between(0, 40)]  # Filter to match IsaacLab replay data since we have more than 40 seconds of data

# --- BASE POSITION ---
plt.figure(figsize=(10, 5))
plt.plot(df['time'], df['base_pos_x'], label='base_pos_x')
plt.plot(df['time'], df['base_pos_y'], label='base_pos_y')
plt.plot(df['time'], df['base_pos_z'], label='base_pos_z')
plt.title("Base Position Over Time")
plt.xlabel("Time [s]")
plt.ylabel("Position [m]")
plt.legend()
plt.grid(True)
if SAVE_FIG: plt.savefig("base_position.png", dpi=200, bbox_inches='tight')

# --- BASE ORIENTATION (QUATERNION) ---
plt.figure(figsize=(10, 5))
plt.plot(df['time'], df['base_quat_w'], label='quat_w')
plt.plot(df['time'], df['base_quat_x'], label='quat_x')
plt.plot(df['time'], df['base_quat_y'], label='quat_y')
plt.plot(df['time'], df['base_quat_z'], label='quat_z')
plt.title("Base Orientation (Quaternion) Over Time")
plt.xlabel("Time [s]")
plt.ylabel("Quaternion Value")
plt.legend()
plt.grid(True)
if SAVE_FIG: plt.savefig("base_orientation.png", dpi=200, bbox_inches='tight')

# --- JOINT POSITIONS ---
joint_pos_cols = [col for col in df.columns if col.startswith('joint_pos_')]
plt.figure(figsize=(12, 6))
for col in joint_pos_cols:
    plt.plot(df['time'], df[col], label=col)
plt.title("Joint Positions Over Time")
plt.xlabel("Time [s]")
plt.ylabel("Joint Position [rad or m]")
plt.legend(ncol=3, fontsize='small')
plt.grid(True)
if SAVE_FIG: plt.savefig("joint_positions.png", dpi=200, bbox_inches='tight')

# --- JOINT VELOCITIES ---
joint_vel_cols = [col for col in df.columns if col.startswith('joint_vel_')]
plt.figure(figsize=(12, 6))
for col in joint_vel_cols:
    plt.plot(df['time'], df[col], label=col)
plt.title("Joint Velocities Over Time")
plt.xlabel("Time [s]")
plt.ylabel("Joint Velocity [rad/s]")
plt.legend(ncol=3, fontsize='small')
plt.grid(True)
if SAVE_FIG: plt.savefig("joint_velocities.png", dpi=200, bbox_inches='tight')

# --- CONTACT FORCES ---
contact_force_cols = [col for col in df.columns if col.startswith('contact_force_')]

# Group by limb (LF, RF, LH, RH)
limbs = sorted(set(col.split('_')[2] for col in contact_force_cols))  # ['LF', 'LH', 'RF', 'RH']

for limb in limbs:
    plt.figure(figsize=(10, 5))
    cols = [col for col in contact_force_cols if f'_{limb}_' in col]
    for col in cols:
        plt.plot(df['time'], df[col], label=col)
    plt.title(f"Contact Forces ({limb}) Over Time")
    plt.xlabel("Time [s]")
    plt.ylabel("Force [N]")
    plt.legend()
    plt.grid(True)
    if SAVE_FIG:
        plt.savefig(f"contact_forces_{limb}.png", dpi=200, bbox_inches='tight')


if SHOW_FIG:
    plt.show()
